In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

def read_from_glue_or_s3(table_name: str, s3_path: str):
    """
    Intenta leer primero desde Glue Catalog.
    Si la tabla no existe o falla la lectura, cae a Parquet en S3.
    """
    try:
        if spark.catalog.tableExists(table_name):
            print(f"Leyendo desde Glue: {table_name}")
            return spark.table(table_name)
        else:
            print(f"No existe en Glue: {table_name}. Leyendo desde S3: {s3_path}")
            return spark.read.parquet(s3_path)
    except Exception as e:
        print(f"Falló lectura desde Glue para {table_name}: {e}")
        print(f"Leyendo desde S3: {s3_path}")
        return spark.read.parquet(s3_path)


df_installments = read_from_glue_or_s3(
    table_name="trusted_db.installments_payments",
    s3_path="s3://hcdr/trusted/installments_payments/"
)

df_previous = read_from_glue_or_s3(
    table_name="trusted_db.previous_application",
    s3_path="s3://hcdr/trusted/previous_application/"
)

df_train = read_from_glue_or_s3(
    table_name="trusted_db.application_train",
    s3_path="s3://hcdr/trusted/application_train/"
)

df_installments = df_installments.dropna(subset=["sk_id_curr", "sk_id_prev"])
df_previous = df_previous.dropna(subset=["sk_id_curr", "sk_id_prev"])

In [ ]:
df_installments.createOrReplaceTempView("installments_raw")

installments_features = spark.sql("""
WITH installments AS (
    SELECT
        *,
        
        -- Diferencia entre fecha real de pago y fecha esperada
        -- > 0 = pagó tarde, < 0 = pagó antes
        (days_entry_payment - days_instalment) AS payment_delay,

        -- Días de atraso: solo valores positivos
        GREATEST(days_entry_payment - days_instalment, 0) AS dpd,

        -- Flag de pago tardío
        CASE
            WHEN GREATEST(days_entry_payment - days_instalment, 0) > 0 THEN 1
            ELSE 0
        END AS late_payment_flag,

        -- Flag de atraso severo: más de 30 días
        CASE
            WHEN GREATEST(days_entry_payment - days_instalment, 0) > 30 THEN 1
            ELSE 0
        END AS late_30_flag,

        -- Flag de pago menor al esperado
        CASE
            WHEN amt_payment < amt_instalment THEN 1
            ELSE 0
        END AS underpaid_flag,

        -- Flag de cuota sin pago registrado
        CASE
            WHEN COALESCE(amt_payment, 0) = 0 THEN 1
            ELSE 0
        END AS missed_payment_flag,

        -- Ratio pagado / esperado
        -- Equivalente a:
        -- amt_payment / amt_instalment
        -- replace([np.inf, -np.inf], np.nan)
        -- clip(upper=3)
        CASE
            WHEN amt_instalment IS NULL OR amt_instalment = 0 THEN NULL
            ELSE LEAST(amt_payment / amt_instalment, 3.0)
        END AS payment_ratio

    FROM installments_raw
)

SELECT
    sk_id_curr,

    -- Número total de registros de cuotas/pagos del cliente
    COUNT(num_instalment_number) AS installments_count,

    -- Número de créditos previos distintos asociados al cliente
    COUNT(DISTINCT sk_id_prev) AS previous_loans_nunique,

    -- Mediana de días de atraso: comportamiento típico del cliente
    percentile_approx(dpd, 0.5) AS dpd_median,

    -- Máximo atraso observado: peor caso del cliente
    MAX(dpd) AS dpd_max,

    -- Proporción de pagos tardíos
    AVG(late_payment_flag) AS late_payment_mean,

    -- Cantidad de pagos con más de 30 días de atraso
    SUM(late_30_flag) AS late_30_sum,

    -- Total que debía pagar
    SUM(amt_instalment) AS amt_instalment_sum,

    -- Total que efectivamente pagó
    SUM(amt_payment) AS amt_payment_sum,

    -- Mediana del ratio pagado / esperado
    percentile_approx(payment_ratio, 0.5) AS payment_ratio_median,

    -- Proporción de veces que pagó menos de lo esperado
    AVG(underpaid_flag) AS underpaid_mean,

    -- Proporción de cuotas sin pago registrado
    AVG(missed_payment_flag) AS missed_payment_mean

FROM installments
GROUP BY sk_id_curr
""")

installments_features.show(5, truncate=False)

In [ ]:
import re

df_previous.createOrReplaceTempView("previous_raw")

# =========================
# 1. NUMÉRICAS
# =========================
previous_num_features = spark.sql("""
WITH previous AS (
    SELECT
        *,

        CASE WHEN amt_annuity IS NULL THEN 1 ELSE 0 END AS amt_annuity_missing_flag,
        CASE WHEN amt_goods_price IS NULL THEN 1 ELSE 0 END AS amt_goods_price_missing_flag,
        CASE WHEN cnt_payment IS NULL THEN 1 ELSE 0 END AS cnt_payment_missing_flag,

        CASE
            WHEN amt_credit > 0 THEN amt_application / amt_credit
            ELSE NULL
        END AS application_credit_ratio,

        CASE
            WHEN amt_credit > 0 THEN amt_annuity / amt_credit
            ELSE NULL
        END AS annuity_credit_ratio,

        CASE
            WHEN amt_credit > 0 THEN amt_goods_price / amt_credit
            ELSE NULL
        END AS goods_credit_ratio,

        CASE
            WHEN COALESCE(sellerplace_area, 0) <= 0 THEN 1
            ELSE 0
        END AS sellerplace_area_nonpositive_flag,

        CASE
            WHEN sellerplace_area > 0 THEN sellerplace_area
            ELSE NULL
        END AS sellerplace_area_positive,

        CASE
            WHEN flag_last_appl_per_contract = 'Y' THEN 1
            WHEN flag_last_appl_per_contract = 'N' THEN 0
            ELSE NULL
        END AS flag_last_appl_per_contract_bin

    FROM previous_raw
)

SELECT
    sk_id_curr,
    COUNT(sk_id_prev) AS previous_count,
    MAX(days_decision) AS days_decision_max,
    percentile_approx(days_decision, 0.5) AS days_decision_median,
    percentile_approx(amt_application, 0.5) AS amt_application_median,
    percentile_approx(amt_credit, 0.5) AS amt_credit_median,
    percentile_approx(amt_annuity, 0.5) AS amt_annuity_median,
    percentile_approx(amt_goods_price, 0.5) AS amt_goods_price_median,
    SUM(amt_credit) AS amt_credit_sum,
    percentile_approx(cnt_payment, 0.5) AS cnt_payment_median,
    percentile_approx(application_credit_ratio, 0.5) AS application_credit_ratio_median,
    percentile_approx(annuity_credit_ratio, 0.5) AS annuity_credit_ratio_median,
    percentile_approx(goods_credit_ratio, 0.5) AS goods_credit_ratio_median,
    AVG(amt_annuity_missing_flag) AS amt_annuity_missing_mean,
    AVG(amt_goods_price_missing_flag) AS amt_goods_price_missing_mean,
    AVG(cnt_payment_missing_flag) AS cnt_payment_missing_mean,
    AVG(flag_last_appl_per_contract_bin) AS last_appl_per_contract_mean,
    AVG(nflag_last_appl_in_day) AS last_appl_in_day_mean,
    AVG(sellerplace_area_nonpositive_flag) AS sellerplace_area_nonpositive_mean,
    percentile_approx(sellerplace_area_positive, 0.5) AS sellerplace_area_positive_median
FROM previous
GROUP BY sk_id_curr
""")

# =========================
# 2. CATEGÓRICAS
# =========================
low_card_cat_cols = [
    "name_client_type",
    "name_product_type",
    "channel_type"
]

def sanitize_col_name(value: str) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-zA-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "unknown"

agg_exprs = []

for col_name in low_card_cat_cols:
    values = (
        df_previous
        .select(col_name)
        .distinct()
        .where(f"{col_name} IS NOT NULL")
        .collect()
    )

    for row in values:
        raw_value = row[col_name]
        safe_value = str(raw_value).replace("'", "''")
        alias = f"{col_name}_{sanitize_col_name(raw_value)}"

        agg_exprs.append(f"""
        AVG(
            CASE
                WHEN {col_name} = '{safe_value}' THEN 1.0
                ELSE 0.0
            END
        ) AS `{alias}`
        """)

cat_sql = f"""
SELECT
    sk_id_curr,
    {", ".join(agg_exprs)}
FROM previous_raw
GROUP BY sk_id_curr
"""

previous_cat_features = spark.sql(cat_sql)

# =========================
# 3. JOIN FINAL
# =========================
previous_features = previous_num_features.join(
    previous_cat_features,
    on="sk_id_curr",
    how="left"
)

previous_features.show(5, truncate=False)

AttributeError: 'DataFrame' object has no attribute 'createOrReplaceTempView'

In [ ]:
import re
from pyspark.sql import functions as F


def prepare_main_application_spark(
    df,
    one_hot: bool = True,
    drop_document_flags: bool = False,
    drop_constant_cols: bool = True,
    view_name: str = "application_raw",
):
    """
    Equivalente Spark SQL de prepare_main_application(...) de pandas.

    Notas:
    - transforma solo columnas existentes
    - hace lower-case de nombres de columnas
    - one-hot y drop_constant_cols requieren generación dinámica
    """

    def qcol(c: str) -> str:
        return f"`{c}`"

    def sanitize_name(value) -> str:
        if value is None:
            return "nan"
        s = str(value).strip().lower()
        s = re.sub(r"[^a-zA-Z0-9]+", "_", s)
        s = re.sub(r"_+", "_", s).strip("_")
        return s or "empty"

    def sql_literal(value):
        if value is None:
            return "NULL"
        if isinstance(value, bool):
            return "TRUE" if value else "FALSE"
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            return str(value)
        return "'" + str(value).replace("'", "''") + "'"

    def cap_expr(col_ref: str, cap_ref: str) -> str:
        return f"""
        CASE
            WHEN {cap_ref} IS NULL THEN {col_ref}
            WHEN {col_ref} IS NULL THEN NULL
            WHEN {col_ref} > {cap_ref} THEN {cap_ref}
            ELSE {col_ref}
        END
        """

    def safe_div_expr(num_ref: str, den_ref: str) -> str:
        return f"""
        CASE
            WHEN ({num_ref}) IS NULL OR ({den_ref}) IS NULL OR ({den_ref}) = 0 THEN NULL
            ELSE ({num_ref}) / ({den_ref})
        END
        """

    def clip01_expr(col_ref: str) -> str:
        return f"LEAST(GREATEST(COALESCE({col_ref}, 0), 0), 1)"

    # 0) lower-case columns
    df_lc = df.toDF(*[c.lower() for c in df.columns])
    cols = df_lc.columns
    cols_set = set(cols)

    df_lc.createOrReplaceTempView(view_name)

    # grupos de columnas
    missing_flag_cols = [
        "obs_30_cnt_social_circle",
        "obs_60_cnt_social_circle",
        "def_30_cnt_social_circle",
        "def_60_cnt_social_circle",
        "ext_source_2",
        "ext_source_3",
        "days_employed",
        "days_last_phone_change",
        "amt_annuity",
        "amt_goods_price",
        "cnt_fam_members",
        "amt_req_credit_bureau_hour",
        "amt_req_credit_bureau_day",
        "amt_req_credit_bureau_week",
        "amt_req_credit_bureau_mon",
        "amt_req_credit_bureau_qrt",
        "amt_req_credit_bureau_year",
    ]
    missing_flag_cols = [c for c in missing_flag_cols if c in cols_set]

    amount_cols = [
        "amt_income_total",
        "amt_credit",
        "amt_annuity",
        "amt_goods_price",
    ]
    amount_cols = [c for c in amount_cols if c in cols_set]

    count_like_cols = [
        "cnt_children",
        "cnt_fam_members",
        "obs_30_cnt_social_circle",
        "def_30_cnt_social_circle",
        "obs_60_cnt_social_circle",
        "def_60_cnt_social_circle",
        "amt_req_credit_bureau_hour",
        "amt_req_credit_bureau_day",
        "amt_req_credit_bureau_week",
        "amt_req_credit_bureau_mon",
        "amt_req_credit_bureau_qrt",
        "amt_req_credit_bureau_year",
    ]
    count_like_cols = [c for c in count_like_cols if c in cols_set]

    bureau_cols = [
        "amt_req_credit_bureau_hour",
        "amt_req_credit_bureau_day",
        "amt_req_credit_bureau_week",
        "amt_req_credit_bureau_mon",
        "amt_req_credit_bureau_qrt",
        "amt_req_credit_bureau_year",
    ]
    bureau_cols = [c for c in bureau_cols if c in cols_set]

    recent_bureau_cols = [
        "amt_req_credit_bureau_hour",
        "amt_req_credit_bureau_day",
        "amt_req_credit_bureau_week",
        "amt_req_credit_bureau_mon",
    ]
    recent_bureau_cols = [c for c in recent_bureau_cols if c in cols_set]

    obs_cols = [c for c in ["obs_30_cnt_social_circle", "obs_60_cnt_social_circle"] if c in cols_set]
    def_cols = [c for c in ["def_30_cnt_social_circle", "def_60_cnt_social_circle"] if c in cols_set]

    contact_cols = [c for c in ["flag_emp_phone", "flag_work_phone", "flag_phone", "flag_email"] if c in cols_set]

    addr_cols = [
        c for c in [
            "reg_region_not_live_region",
            "reg_region_not_work_region",
            "live_region_not_work_region",
            "reg_city_not_live_city",
            "reg_city_not_work_city",
            "live_city_not_work_city",
        ] if c in cols_set
    ]

    doc_cols = [c for c in cols if c.startswith("flag_document_")]

    low_card_cat_cols = [
        "name_contract_type",
        "code_gender",
        "flag_own_car",
        "flag_own_realty",
        "name_income_type",
        "name_education_type",
        "name_family_status",
        "name_housing_type",
        "weekday_appr_process_start",
    ]
    low_card_cat_cols = [c for c in low_card_cat_cols if c in cols_set]

    ext_cols = [c for c in ["ext_source_1", "ext_source_2", "ext_source_3"] if c in cols_set]

    # 1) stage0: base + missing flags + limpieza anomalías en days_*
    stage0_select = []
    stage0_cols = []

    for c in cols:
        if c == "days_birth":
            cond = "(b.`days_birth` >= 0 OR b.`days_birth` = 365243)"
            stage0_select.append(f"CASE WHEN {cond} THEN NULL ELSE b.`days_birth` END AS `days_birth`")
            stage0_select.append(f"CASE WHEN {cond} THEN 1 ELSE 0 END AS `days_birth_anom_flag`")
            stage0_select.append(f"-(CASE WHEN {cond} THEN NULL ELSE b.`days_birth` END) / 365.0 AS `age_years`")
            stage0_cols.extend(["days_birth", "days_birth_anom_flag", "age_years"])

        elif c == "days_employed":
            cond = "(b.`days_employed` > 0 OR b.`days_employed` = 365243)"
            stage0_select.append(f"CASE WHEN {cond} THEN NULL ELSE b.`days_employed` END AS `days_employed`")
            stage0_select.append(f"CASE WHEN {cond} THEN 1 ELSE 0 END AS `days_employed_anom_flag`")
            stage0_select.append(f"-(CASE WHEN {cond} THEN NULL ELSE b.`days_employed` END) / 365.0 AS `employment_years`")
            stage0_cols.extend(["days_employed", "days_employed_anom_flag", "employment_years"])

        elif c == "days_registration":
            cond = "(b.`days_registration` > 0 OR b.`days_registration` = 365243)"
            stage0_select.append(f"CASE WHEN {cond} THEN NULL ELSE b.`days_registration` END AS `days_registration`")
            stage0_select.append(f"CASE WHEN {cond} THEN 1 ELSE 0 END AS `days_registration_anom_flag`")
            stage0_select.append(f"-(CASE WHEN {cond} THEN NULL ELSE b.`days_registration` END) / 365.0 AS `registration_years`")
            stage0_cols.extend(["days_registration", "days_registration_anom_flag", "registration_years"])

        elif c == "days_id_publish":
            cond = "(b.`days_id_publish` > 0 OR b.`days_id_publish` = 365243)"
            stage0_select.append(f"CASE WHEN {cond} THEN NULL ELSE b.`days_id_publish` END AS `days_id_publish`")
            stage0_select.append(f"CASE WHEN {cond} THEN 1 ELSE 0 END AS `days_id_publish_anom_flag`")
            stage0_select.append(f"-(CASE WHEN {cond} THEN NULL ELSE b.`days_id_publish` END) / 365.0 AS `id_publish_years`")
            stage0_cols.extend(["days_id_publish", "days_id_publish_anom_flag", "id_publish_years"])

        elif c == "days_last_phone_change":
            cond = "(b.`days_last_phone_change` > 0 OR b.`days_last_phone_change` = 365243)"
            stage0_select.append(f"CASE WHEN {cond} THEN NULL ELSE b.`days_last_phone_change` END AS `days_last_phone_change`")
            stage0_select.append(f"CASE WHEN {cond} THEN 1 ELSE 0 END AS `days_last_phone_change_anom_flag`")
            stage0_select.append(f"-(CASE WHEN {cond} THEN NULL ELSE b.`days_last_phone_change` END) / 365.0 AS `last_phone_change_years`")
            stage0_cols.extend(["days_last_phone_change", "days_last_phone_change_anom_flag", "last_phone_change_years"])

        else:
            stage0_select.append(f"b.{qcol(c)} AS {qcol(c)}")
            stage0_cols.append(c)

    for c in missing_flag_cols:
        stage0_select.append(f"CASE WHEN b.{qcol(c)} IS NULL THEN 1 ELSE 0 END AS {qcol(c + '_missing_flag')}")
        stage0_cols.append(c + "_missing_flag")

    stage0_sql = f"""
    stage0 AS (
        SELECT
            {", ".join(stage0_select)}
        FROM {view_name} b
    )
    """

    # 2) caps globales
    cap_agg_exprs = []

    for c in amount_cols:
        cap_agg_exprs.append(f"percentile_approx(s0.{qcol(c)}, 0.995, 10000) AS {qcol(c + '_p995')}")

    for c in count_like_cols:
        cap_agg_exprs.append(f"percentile_approx(s0.{qcol(c)}, 0.99, 10000) AS {qcol(c + '_p99')}")

    caps_sql = f"""
    caps AS (
        SELECT
            {", ".join(cap_agg_exprs) if cap_agg_exprs else "1 AS `_dummy`"}
        FROM stage0 s0
    )
    """

    # 3) stage1: cap/log + ratios raw + features fila a fila
    stage1_select = [f"s0.{qcol(c)} AS {qcol(c)}" for c in stage0_cols]
    stage1_cols = list(stage0_cols)

    non_ratio_feature_cols = []
    ratio_raw_to_final = {}

    # amount cap + log
    for c in amount_cols:
        capref = f"caps.{qcol(c + '_p995')}"
        capped = cap_expr(f"s0.{qcol(c)}", capref)
        stage1_select.append(f"{capped} AS {qcol(c + '_cap')}")
        stage1_select.append(f"CASE WHEN ({capped}) IS NULL THEN NULL ELSE log1p({capped}) END AS {qcol(c + '_log')}")
        stage1_cols.extend([c + "_cap", c + "_log"])
        non_ratio_feature_cols.extend([c + "_cap", c + "_log"])

    # count-like cap
    for c in count_like_cols:
        capref = f"caps.{qcol(c + '_p99')}"
        capped = cap_expr(f"s0.{qcol(c)}", capref)
        stage1_select.append(f"{capped} AS {qcol(c + '_cap')}")
        stage1_cols.append(c + "_cap")
        non_ratio_feature_cols.append(c + "_cap")

    # ratios raw
    if {"amt_credit", "amt_income_total"}.issubset(stage0_cols):
        stage1_select.append(
            f"{safe_div_expr('s0.`amt_credit`', '(s0.`amt_income_total` + 1)')} AS `credit_income_ratio_raw`"
        )
        stage1_cols.append("credit_income_ratio_raw")
        ratio_raw_to_final["credit_income_ratio_raw"] = "credit_income_ratio"

    if {"amt_annuity", "amt_income_total"}.issubset(stage0_cols):
        stage1_select.append(
            f"{safe_div_expr('s0.`amt_annuity`', '(s0.`amt_income_total` + 1)')} AS `annuity_income_ratio_raw`"
        )
        stage1_cols.append("annuity_income_ratio_raw")
        ratio_raw_to_final["annuity_income_ratio_raw"] = "annuity_income_ratio"

    if {"amt_annuity", "amt_credit"}.issubset(stage0_cols):
        stage1_select.append(
            f"{safe_div_expr('s0.`amt_annuity`', '(s0.`amt_credit` + 1)')} AS `annuity_credit_ratio_raw`"
        )
        stage1_cols.append("annuity_credit_ratio_raw")
        ratio_raw_to_final["annuity_credit_ratio_raw"] = "annuity_credit_ratio"

    if {"amt_goods_price", "amt_credit"}.issubset(stage0_cols):
        stage1_select.append(
            f"{safe_div_expr('s0.`amt_goods_price`', '(s0.`amt_credit` + 1)')} AS `goods_credit_ratio_raw`"
        )
        stage1_cols.append("goods_credit_ratio_raw")
        ratio_raw_to_final["goods_credit_ratio_raw"] = "goods_credit_ratio"

    if {"amt_income_total", "cnt_fam_members"}.issubset(stage0_cols):
        stage1_select.append(
            f"{safe_div_expr('s0.`amt_income_total`', '(COALESCE(s0.`cnt_fam_members`, 1) + 1)')} AS `income_per_family_member_raw`"
        )
        stage1_cols.append("income_per_family_member_raw")
        ratio_raw_to_final["income_per_family_member_raw"] = "income_per_family_member"

    if {"amt_income_total", "cnt_children"}.issubset(stage0_cols):
        stage1_select.append(
            f"{safe_div_expr('s0.`amt_income_total`', '(s0.`cnt_children` + 1)')} AS `income_per_child_raw`"
        )
        stage1_cols.append("income_per_child_raw")
        ratio_raw_to_final["income_per_child_raw"] = "income_per_child"

    if {"employment_years", "age_years"}.issubset(stage0_cols):
        stage1_select.append(
            f"{safe_div_expr('s0.`employment_years`', '(s0.`age_years` + 1e-6)')} AS `employment_age_ratio_raw`"
        )
        stage1_cols.append("employment_age_ratio_raw")
        ratio_raw_to_final["employment_age_ratio_raw"] = "employment_age_ratio"

    if {"cnt_children", "cnt_fam_members"}.issubset(stage0_cols):
        stage1_select.append(
            f"{safe_div_expr('s0.`cnt_children`', '(COALESCE(s0.`cnt_fam_members`, 1) + 1e-6)')} AS `children_family_ratio_raw`"
        )
        stage1_cols.append("children_family_ratio_raw")
        ratio_raw_to_final["children_family_ratio_raw"] = "children_family_ratio"

    # EXT_SOURCE
    if len(ext_cols) >= 2:
        n_expr = " + ".join([f"CASE WHEN s0.{qcol(c)} IS NOT NULL THEN 1 ELSE 0 END" for c in ext_cols])
        sum_expr = " + ".join([f"COALESCE(s0.{qcol(c)}, 0.0)" for c in ext_cols])
        sumsq_expr = " + ".join([f"COALESCE(POWER(s0.{qcol(c)}, 2.0), 0.0)" for c in ext_cols])
        arr_expr = ", ".join([f"s0.{qcol(c)}" for c in ext_cols])

        stage1_select.append(
            f"CASE WHEN ({n_expr}) > 0 THEN ({sum_expr}) / ({n_expr}) ELSE NULL END AS `ext_sources_mean`"
        )
        stage1_select.append(
            f"CASE WHEN ({n_expr}) > 0 THEN array_min(filter(array({arr_expr}), x -> x is not null)) ELSE NULL END AS `ext_sources_min`"
        )
        stage1_select.append(
            f"CASE WHEN ({n_expr}) > 0 THEN array_max(filter(array({arr_expr}), x -> x is not null)) ELSE NULL END AS `ext_sources_max`"
        )
        stage1_select.append(
            f"""
            CASE
                WHEN ({n_expr}) > 1 THEN
                    SQRT(
                        GREATEST(
                            (({sumsq_expr}) - POWER(({sum_expr}), 2.0) / ({n_expr})) / (({n_expr}) - 1),
                            0.0
                        )
                    )
                ELSE NULL
            END AS `ext_sources_std`
            """
        )
        stage1_cols.extend(["ext_sources_mean", "ext_sources_min", "ext_sources_max", "ext_sources_std"])
        non_ratio_feature_cols.extend(["ext_sources_mean", "ext_sources_min", "ext_sources_max", "ext_sources_std"])

    # Bureau requests
    if bureau_cols:
        total_expr = " + ".join([f"COALESCE(s0.{qcol(c)}, 0)" for c in bureau_cols])
        stage1_select.append(f"{total_expr} AS `bureau_requests_total`")
        stage1_cols.append("bureau_requests_total")
        non_ratio_feature_cols.append("bureau_requests_total")

    if recent_bureau_cols:
        recent_expr = " + ".join([f"COALESCE(s0.{qcol(c)}, 0)" for c in recent_bureau_cols])
        stage1_select.append(f"{recent_expr} AS `bureau_requests_recent`")
        stage1_cols.append("bureau_requests_recent")
        non_ratio_feature_cols.append("bureau_requests_recent")

    # Social circle
    obs_expr = None
    def_expr = None

    if obs_cols:
        obs_expr = " + ".join([f"COALESCE(s0.{qcol(c)}, 0)" for c in obs_cols])
        stage1_select.append(f"{obs_expr} AS `social_obs_total`")
        stage1_cols.append("social_obs_total")
        non_ratio_feature_cols.append("social_obs_total")

    if def_cols:
        def_expr = " + ".join([f"COALESCE(s0.{qcol(c)}, 0)" for c in def_cols])
        stage1_select.append(f"{def_expr} AS `social_def_total`")
        stage1_cols.append("social_def_total")
        non_ratio_feature_cols.append("social_def_total")

    if obs_expr is not None and def_expr is not None:
        stage1_select.append(
            f"{safe_div_expr(f'({def_expr})', f'(({obs_expr}) + 1)')} AS `social_def_ratio_raw`"
        )
        stage1_cols.append("social_def_ratio_raw")
        ratio_raw_to_final["social_def_ratio_raw"] = "social_def_ratio"

    # Contactabilidad
    if contact_cols:
        contact_expr = " + ".join([clip01_expr(f"s0.{qcol(c)}") for c in contact_cols])
        stage1_select.append(f"{contact_expr} AS `num_contact_channels`")
        stage1_cols.append("num_contact_channels")
        non_ratio_feature_cols.append("num_contact_channels")

    # Direcciones
    if addr_cols:
        addr_expr = " + ".join([clip01_expr(f"s0.{qcol(c)}") for c in addr_cols])
        stage1_select.append(f"{addr_expr} AS `address_mismatch_count`")
        stage1_cols.append("address_mismatch_count")
        non_ratio_feature_cols.append("address_mismatch_count")

    # Documentos
    if doc_cols:
        docs_expr = " + ".join([clip01_expr(f"s0.{qcol(c)}") for c in doc_cols])
        stage1_select.append(f"{docs_expr} AS `num_documents_provided`")
        stage1_cols.append("num_documents_provided")
        non_ratio_feature_cols.append("num_documents_provided")

    stage1_sql = f"""
    stage1 AS (
        SELECT
            {", ".join(stage1_select)}
        FROM stage0 s0
        CROSS JOIN caps
    )
    """

    # 4) caps de ratios
    ratio_cap_exprs = [
        f"percentile_approx(s1.{qcol(raw_col)}, 0.995, 10000) AS {qcol(raw_col + '_p995')}"
        for raw_col in ratio_raw_to_final.keys()
    ]

    ratio_caps_sql = f"""
    ratio_caps AS (
        SELECT
            {", ".join(ratio_cap_exprs) if ratio_cap_exprs else "1 AS `_dummy`"}
        FROM stage1 s1
    )
    """

    # 5) stage2 final sin dummies
    stage2_select = []
    final_feature_cols = []

    for c in stage0_cols:
        stage2_select.append(f"s1.{qcol(c)} AS {qcol(c)}")
        final_feature_cols.append(c)

    for c in non_ratio_feature_cols:
        stage2_select.append(f"s1.{qcol(c)} AS {qcol(c)}")
        final_feature_cols.append(c)

    for raw_col, final_col in ratio_raw_to_final.items():
        capped = cap_expr(f"s1.{qcol(raw_col)}", f"ratio_caps.{qcol(raw_col + '_p995')}")
        stage2_select.append(f"{capped} AS {qcol(final_col)}")
        final_feature_cols.append(final_col)

    stage2_sql = f"""
    stage2 AS (
        SELECT
            {", ".join(stage2_select)}
        FROM stage1 s1
        CROSS JOIN ratio_caps
    )
    """

    # 6) one-hot / drop docs
    final_base_cols = list(final_feature_cols)

    if drop_document_flags and doc_cols:
        final_base_cols = [c for c in final_base_cols if c not in doc_cols]

    dummy_exprs = []
    if one_hot and low_card_cat_cols:
        final_base_cols = [c for c in final_base_cols if c not in low_card_cat_cols]

        for c in low_card_cat_cols:
            distinct_vals = (
                df_lc.select(c)
                .distinct()
                .collect()
            )

            for row in distinct_vals:
                raw_val = row[c]
                if raw_val is None:
                    continue
                alias = f"{c}_{sanitize_name(raw_val)}"
                lit = sql_literal(raw_val)
                dummy_exprs.append(
                    f"CASE WHEN s2.{qcol(c)} = {lit} THEN 1 ELSE 0 END AS {qcol(alias)}"
                )

            # dummy_na=True
            dummy_exprs.append(
                f"CASE WHEN s2.{qcol(c)} IS NULL THEN 1 ELSE 0 END AS {qcol(c + '_nan')}"
            )

    final_select_exprs = [f"s2.{qcol(c)} AS {qcol(c)}" for c in final_base_cols] + dummy_exprs

    final_sql = f"""
    WITH
    {stage0_sql},
    {caps_sql},
    {stage1_sql},
    {ratio_caps_sql},
    {stage2_sql}
    SELECT
        {", ".join(final_select_exprs)}
    FROM stage2 s2
    """

    result_df = spark.sql(final_sql)

    # 7) drop constant cols
    if drop_constant_cols and result_df.columns:
        distinct_exprs = [
            F.countDistinct(
                F.coalesce(F.col(c).cast("string"), F.lit("__NULL__"))
            ).alias(c)
            for c in result_df.columns
        ]

        nunique_row = result_df.agg(*distinct_exprs).collect()[0].asDict()
        constant_cols = [c for c, n in nunique_row.items() if n <= 1]

        if constant_cols:
            result_df = result_df.drop(*constant_cols)

    return result_df

   sk_id_curr  previous_count  days_decision_max  days_decision_median  \
0    100001.0               1            -1740.0               -1740.0   
1    100002.0               1             -606.0                -606.0   
2    100003.0               3             -746.0                -828.0   
3    100004.0               1             -815.0                -815.0   
4    100005.0               2             -315.0                -536.0   

   amt_application_median  amt_credit_median  amt_annuity_median  \
0                24835.50           23787.00            3951.000   
1               179055.00          179055.00            9251.775   
2               337500.00          348637.50           64567.665   
3                24282.00           20106.00            5357.250   
4                22308.75           20076.75            4813.200   

   amt_goods_price_median  amt_credit_sum  cnt_payment_median  ...  \
0                 24835.5         23787.0                 8.0  ...   
1     

In [ ]:
train_prepared = prepare_main_application_spark(
    df_train,
    one_hot=True,
    drop_document_flags=False,
    drop_constant_cols=True,
    view_name="train_raw"
)

train_prepared.show(5, truncate=False)

In [ ]:
final_features = (
    train_prepared
    .join(previous_features, on="sk_id_curr", how="left")
    .join(installments_features, on="sk_id_curr", how="left")
)

In [ ]:
final_features.write.mode("overwrite").parquet(
    "s3://hcdr/refined/train_final_features/"
)

In [ ]:
from pyspark.sql import functions as F

def encode_remaining_categoricals(
    df,
    id_cols=("sk_id_curr",),
    target_cols=("target",),
    max_ohe_cardinality=12,
    drop_original_string_cols=True,
    drop_index_cols=True,
):
    """
    Codifica solo las columnas que aún no están codificadas.

    Regla:
    - columnas string -> candidatas a codificación
    - si cardinalidad <= max_ohe_cardinality -> one-hot wide
    - si cardinalidad > max_ohe_cardinality -> label encoding (índice)
    - columnas numéricas ya existentes no se tocan

    Devuelve:
    - df_out
    - metadata con qué columnas fueron OHE vs index
    """

    protected_cols = set(id_cols) | set(target_cols)

    # 1) detectar columnas categóricas que aún no están codificadas
    string_cols = [c for c, t in df.dtypes if t == "string" and c not in protected_cols]

    if not string_cols:
        return df, {
            "string_cols_detected": [],
            "ohe_cols": [],
            "indexed_only_cols": [],
        }

    # 2) cardinalidad por columna
    cardinality_exprs = [
        F.countDistinct(F.col(c)).alias(c)
        for c in string_cols
    ]
    cardinalities = df.agg(*cardinality_exprs).collect()[0].asDict()

    ohe_cols = [c for c in string_cols if cardinalities[c] <= max_ohe_cardinality]
    indexed_only_cols = [c for c in string_cols if cardinalities[c] > max_ohe_cardinality]

    df_out = df

    # 3) ONE-HOT "wide" para baja cardinalidad
    # genera columnas tipo col_valor = 0/1
    for c in ohe_cols:
        values = (
            df_out
            .select(c)
            .distinct()
            .collect()
        )

        for row in values:
            raw_val = row[c]

            if raw_val is None:
                alias = f"{c}_nan"
                df_out = df_out.withColumn(
                    alias,
                    F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0))
                )
            else:
                safe_name = (
                    str(raw_val)
                    .strip()
                    .lower()
                    .replace(" ", "_")
                    .replace("/", "_")
                    .replace("-", "_")
                    .replace(".", "_")
                    .replace(",", "_")
                    .replace("(", "")
                    .replace(")", "")
                )
                while "__" in safe_name:
                    safe_name = safe_name.replace("__", "_")
                safe_name = safe_name.strip("_")
                if not safe_name:
                    safe_name = "empty"

                alias = f"{c}_{safe_name}"

                df_out = df_out.withColumn(
                    alias,
                    F.when(F.col(c) == F.lit(raw_val), F.lit(1)).otherwise(F.lit(0))
                )

        if drop_original_string_cols:
            df_out = df_out.drop(c)

    # 4) LABEL ENCODING para alta cardinalidad
    # lo hago sin MLlib para dejar columnas escalares, no vectores
    for c in indexed_only_cols:
        mapping_rows = (
            df_out
            .select(c)
            .distinct()
            .orderBy(c)
            .collect()
        )

        mapping = []
        idx = 0
        null_seen = False

        for row in mapping_rows:
            val = row[c]
            if val is None:
                null_seen = True
            else:
                mapping.append((val, float(idx)))
                idx += 1

        map_col = None
        if mapping:
            flat = []
            for k, v in mapping:
                flat.extend([F.lit(k), F.lit(v)])
            map_col = F.create_map(*flat)

        new_col = f"{c}_idx"

        if map_col is not None:
            df_out = df_out.withColumn(
                new_col,
                F.when(F.col(c).isNull(), F.lit(float(idx)) if null_seen else F.lit(None).cast("double"))
                 .otherwise(map_col[F.col(c)])
            )
        else:
            df_out = df_out.withColumn(new_col, F.lit(None).cast("double"))

        if drop_original_string_cols:
            df_out = df_out.drop(c)

    if drop_index_cols:
        # no hace nada extra aquí porque el índice final ya es la columna escalar *_idx
        pass

    return df_out, {
        "string_cols_detected": string_cols,
        "ohe_cols": ohe_cols,
        "indexed_only_cols": indexed_only_cols,
        "cardinalities": cardinalities,
    }

In [ ]:
final_features_encoded, encoding_meta = encode_remaining_categoricals(
    final_features,
    id_cols=("sk_id_curr",),
    target_cols=("target",),
    max_ohe_cardinality=12,
    drop_original_string_cols=True
)

print("Columnas string detectadas:", encoding_meta["string_cols_detected"])
print("One-hot:", encoding_meta["ohe_cols"])
print("Indexadas:", encoding_meta["indexed_only_cols"])

final_features_encoded.printSchema()
final_features_encoded.show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Base
df_model = final_features_encoded

# Columnas que NO van al modelo
exclude_cols = {"sk_id_curr", "target"}

# Solo features numéricas
numeric_types = {
    "int", "bigint", "double", "float", "smallint", "tinyint", "decimal", "long", "short"
}

feature_cols = [
    c for c, t in df_model.dtypes
    if c not in exclude_cols and any(t.startswith(nt) for nt in numeric_types)
]

print(f"Número de features: {len(feature_cols)}")

# Relleno simple de nulls para evitar problemas en el ensamblado/modelo
df_model = df_model.fillna(0, subset=feature_cols)

# Vector de features
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

df_vec = assembler.transform(df_model).select("target", "features")

# Split train/test
train_df, test_df = df_vec.randomSplit([0.8, 0.2], seed=42)

# Modelo
rf = RandomForestClassifier(
    labelCol="target",
    featuresCol="features",
    numTrees=20,
    maxDepth=3,
    seed=42
)

rf_model = rf.fit(train_df)

# Evaluación
pred = rf_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(
    labelCol="target",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(pred)
print("AUC test:", auc)

# Feature importance
importances = rf_model.featureImportances.toArray().tolist()

importance_rows = list(zip(feature_cols, importances))
importance_df = spark.createDataFrame(importance_rows, ["feature", "importance"]) \
    .orderBy(F.desc("importance"))

importance_df.show(50, truncate=False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import ChiSquareTest

def quick_feature_ranking(df, target_col="target", id_cols=("sk_id_curr",)):
    protected = set(id_cols) | {target_col}

    numeric_prefixes = ("tinyint", "smallint", "int", "bigint", "float", "double", "decimal", "long", "short")
    feature_cols = [
        c for c, t in df.dtypes
        if c not in protected and any(t.startswith(p) for p in numeric_prefixes)
    ]

    work = df.select([target_col] + list(id_cols) + feature_cols).fillna(0, subset=feature_cols)

    # 1) detectar binarias/dummies vs continuas
    agg_exprs = []
    for c in feature_cols:
        agg_exprs.extend([
            F.min(F.col(c)).alias(f"{c}__min"),
            F.max(F.col(c)).alias(f"{c}__max"),
            F.countDistinct(F.col(c)).alias(f"{c}__nuniq"),
        ])

    stats = work.agg(*agg_exprs).collect()[0].asDict()

    binary_cols = []
    continuous_cols = []

    for c in feature_cols:
        cmin = stats[f"{c}__min"]
        cmax = stats[f"{c}__max"]
        nuniq = stats[f"{c}__nuniq"]

        if cmin is not None and cmax is not None and cmin >= 0 and cmax <= 1 and nuniq <= 2:
            binary_cols.append(c)
        else:
            continuous_cols.append(c)

    # 2) ranking rápido para continuas: |corr Pearson con target|
    cont_rows = []
    for c in continuous_cols:
        corr_val = work.stat.corr(c, target_col)
        cont_rows.append((c, corr_val, abs(corr_val) if corr_val is not None else None))

    cont_rank = (
        spark.createDataFrame(cont_rows, ["feature", "pearson_corr", "abs_pearson_corr"])
        .orderBy(F.desc("abs_pearson_corr"))
    )

    # 3) ranking para binarias/dummies: Chi-square
    if binary_cols:
        assembler = VectorAssembler(
            inputCols=binary_cols,
            outputCol="features",
            handleInvalid="keep"
        )

        chi_df = assembler.transform(work).select(
            F.col(target_col).cast("double").alias("label"),
            "features"
        )

        chi_res = ChiSquareTest.test(
            chi_df,
            featuresCol="features",
            labelCol="label",
            flatten=True
        )

        index_map = spark.createDataFrame(
            [(i, c) for i, c in enumerate(binary_cols)],
            ["featureIndex", "feature"]
        )

        binary_rank = (
            chi_res.join(index_map, on="featureIndex", how="left")
            .withColumn("neg_log10_pvalue", -F.log10(F.greatest(F.col("pValue"), F.lit(1e-300))))
            .select("feature", "statistic", "pValue", "neg_log10_pvalue")
            .orderBy(F.asc("pValue"), F.desc("statistic"))
        )
    else:
        binary_rank = spark.createDataFrame(
            [],
            "feature string, statistic double, pValue double, neg_log10_pvalue double"
        )

    return {
        "binary_cols": binary_cols,
        "continuous_cols": continuous_cols,
        "binary_rank": binary_rank,
        "continuous_rank": cont_rank,
    }

In [ ]:
ranking = quick_feature_ranking(
    final_features_encoded,
    target_col="target",
    id_cols=("sk_id_curr",)
)

print("Binarias:", len(ranking["binary_cols"]))
print("Continuas:", len(ranking["continuous_cols"]))

ranking["binary_rank"].show(50, truncate=False)
ranking["continuous_rank"].show(50, truncate=False)

In [ ]:
base_path = "s3://hcdr/refined/feature_selection"

importance_df.write \
    .mode("overwrite") \
    .parquet(f"{base_path}/importance_df/")

ranking["binary_rank"].write \
    .mode("overwrite") \
    .parquet(f"{base_path}/binary_rank/")

ranking["continuous_rank"].write \
    .mode("overwrite") \
    .parquet(f"{base_path}/continuous_rank/")